# **Phase 2: Design decisions**

In [2]:
import duckdb
import time
import psutil
import os
import time
import glob
import tempfile

**2A - Prove the two-stage pipeline**

In [3]:
# Stage 1: Parser catches this — doesn't even look at any data
try:
    duckdb.sql("SELCT * FROM flights.csv")
except duckdb.ParserException as e:
    print(f"[STAGE 1 - PARSER] Caught before catalog lookup:\n{e}\n")

# Stage 2: Parser passes this, Binder catches it — table doesn't exist
try:
    duckdb.sql("SELECT * FROM nonexistent_table")
except duckdb.CatalogException as e:
    print(f"[STAGE 2 - BINDER/CATALOG] Parser passed, catalog consulted:\n{e}\n")

# Stage 3: Both pass, Binder catches wrong column
try:
    duckdb.sql("SELECT DEP_DELAY FROM read_csv_auto('flights.csv')")
except duckdb.BinderException as e:
    print(f"[STAGE 3 - BINDER] Table found, column resolution failed:\n{e}\n")

[STAGE 1 - PARSER] Caught before catalog lookup:
Parser Error: syntax error at or near "SELCT"

LINE 1: SELCT * FROM flights.csv
        ^

[STAGE 2 - BINDER/CATALOG] Parser passed, catalog consulted:
Catalog Error: Table with name nonexistent_table does not exist!
Did you mean "sqlite_temp_master"?

[STAGE 3 - BINDER] Table found, column resolution failed:
Binder Error: Referenced column "DEP_DELAY" not found in FROM clause!
Candidate bindings: "DEPARTURE_DELAY", "ELAPSED_TIME", "AIRLINE_DELAY", "WEATHER_DELAY", "AIR_SYSTEM_DELAY"



**2B - Vectorized execution trend**

In [4]:
query = """
    SELECT AIRLINE, count(*) as delayed_flights, avg(DEPARTURE_DELAY) as avg_delay
    FROM read_csv_auto('flights.csv')
    WHERE DEPARTURE_DELAY > 15
    GROUP BY AIRLINE
    ORDER BY delayed_flights DESC
"""

process = psutil.Process(os.getpid())

for vec_size in [2048, 1024, 512, 128, 32, 2]:
    con = duckdb.connect()
    con.execute("SET threads=1")

    # CPU time snapshot before
    cpu_before = process.cpu_times()
    start = time.time()

    con.execute(query).fetchall()

    # CPU time snapshot after
    elapsed = time.time() - start
    cpu_after = process.cpu_times()
    cpu_user = cpu_after.user - cpu_before.user

    print(f"Vector size {vec_size:>5} | Wall: {elapsed:.3f}s | CPU user: {cpu_user:.3f}s")
    con.close()

Vector size  2048 | Wall: 6.240s | CPU user: 5.047s
Vector size  1024 | Wall: 5.454s | CPU user: 4.500s
Vector size   512 | Wall: 5.078s | CPU user: 4.453s
Vector size   128 | Wall: 5.646s | CPU user: 4.688s
Vector size    32 | Wall: 4.858s | CPU user: 4.156s
Vector size     2 | Wall: 4.080s | CPU user: 3.844s


**2C - Buffer Manager spill**

In [5]:
SPILL_DIR = os.path.abspath("./duckdb_spill_test")
os.makedirs(SPILL_DIR, exist_ok=True)

query = """
    SELECT AIRLINE, count(*) as delayed_flights, avg(DEPARTURE_DELAY) as avg_delay
    FROM read_csv_auto('flights.csv')
    WHERE DEPARTURE_DELAY > 15
    GROUP BY AIRLINE
    ORDER BY delayed_flights DESC
"""

con = duckdb.connect()

# Confirm DuckDB actually registered the spill directory
con.execute(f"SET temp_directory='{SPILL_DIR}'")
con.execute("SET threads=1")
con.execute("SET memory_limit='300MB'")
con.execute("SET preserve_insertion_order=false")

# Check what DuckDB thinks the temp dir is
result = con.execute("SELECT current_setting('temp_directory')").fetchone()
print(f"DuckDB temp_directory is set to: {result[0]}")

t0 = time.time()
con.execute(query).fetchall()
elapsed = time.time() - t0
print(f"Query completed in: {elapsed:.3f}s")

# Now scan the directory with ALL file types, not just .tmp
print(f"\nAll files in spill dir ({SPILL_DIR}):")
for root, dirs, files in os.walk(SPILL_DIR):
    for f in files:
        full_path = os.path.join(root, f)
        print(f"  {full_path}  →  {os.path.getsize(full_path)/1024:.1f} KB")

if not any(os.scandir(SPILL_DIR)):
    print("  (directory is empty — checking system temp dir)")
    sys_temp = tempfile.gettempdir()
    print(f"\nSystem temp dir: {sys_temp}")
    for f in os.listdir(sys_temp):
        if 'duck' in f.lower() or f.endswith('.tmp'):
            full = os.path.join(sys_temp, f)
            print(f"  {full}  →  {os.path.getsize(full)/1024:.1f} KB")

con.close()

DuckDB temp_directory is set to: d:\Sem_2\BDE\Course_Project\duckdb_spill_test
Query completed in: 4.679s

All files in spill dir (d:\Sem_2\BDE\Course_Project\duckdb_spill_test):
  (directory is empty — checking system temp dir)

System temp dir: C:\Users\RAMPRA~1\AppData\Local\Temp
  C:\Users\RAMPRA~1\AppData\Local\Temp\08017220-ecd0-43ad-929b-9dbd3aeb35cb.tmp  →  0.0 KB
  C:\Users\RAMPRA~1\AppData\Local\Temp\0a3ea0bc-4941-4118-8262-9939252424c8.tmp  →  0.0 KB
  C:\Users\RAMPRA~1\AppData\Local\Temp\0d1de8f3-30d7-4789-a0c2-0504d3b91c57.tmp  →  0.0 KB
  C:\Users\RAMPRA~1\AppData\Local\Temp\2968bca9-9d53-43b3-be27-4fb981378e49.tmp  →  0.0 KB
  C:\Users\RAMPRA~1\AppData\Local\Temp\2a2abc25-bb17-43c1-bdce-e5b10da7862b.tmp  →  0.0 KB
  C:\Users\RAMPRA~1\AppData\Local\Temp\2b8f1f4c-fa64-4f83-8b50-0f366b6a2f6d.tmp  →  0.0 KB
  C:\Users\RAMPRA~1\AppData\Local\Temp\3412ed60-d54d-4eb7-b24c-60d2c3c30da2.tmp  →  0.0 KB
  C:\Users\RAMPRA~1\AppData\Local\Temp\35aa5aea-eba3-42dc-ac1f-cdf631cf39e0.tmp

In [6]:
sys_temp = r"C:\Users\RAMPRA~1\AppData\Local\Temp"
duckdb_files = [f for f in os.listdir(sys_temp) 
                if f.startswith('tmp') and f.endswith('.tmp') 
                and len(f) <= 12]  # short Windows names = DuckDB pattern

total_mb = 0
print("DuckDB spill files detected:")
print(f"{'File':<20} {'Size':>10}")
print("-" * 32)
for f in sorted(duckdb_files):
    full = os.path.join(sys_temp, f)
    size_mb = os.path.getsize(full) / (1024*1024)
    total_mb += size_mb
    print(f"{f:<20} {size_mb:>8.1f} MB")
print("-" * 32)
print(f"{'Total spill:':<20} {total_mb:>8.1f} MB")

DuckDB spill files detected:
File                       Size
--------------------------------
tmp2867.tmp              17.5 MB
tmp2AAF.tmp              17.5 MB
tmp2B64.tmp              17.1 MB
tmp4A8B.tmp              17.1 MB
tmp8042.tmp              17.1 MB
tmp9852.tmp              17.1 MB
tmpA304.tmp              17.5 MB
tmpF749.tmp              17.1 MB
--------------------------------
Total spill:            138.0 MB
